# Lecture 7 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> **Push to:** `week07/lecture07_exercise.ipynb`


In [4]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Global Energy Mix by Country and Source
df = pd.read_csv('global_energy_mix.csv')


# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))


Loaded: 103 rows
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         58  6929.1   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting
- Use `path=['Region', 'Country', 'Source']` for the hierarchy
- Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
- Show TWh values in labels — no percentages
- Grey out parent nodes (Region and Country level)
- Insight title naming which region or country is most fossil-dependent

> 💡 `df.loc[df['Source_Type'] == 'Fossil']`


In [5]:
# Task 1
# YOUR CODE HERE
df_fossil = df[df["Source_Type"] == "Fossil"]

fig = px.treemap(
    df_fossil,
    path=["Region", "Country", "Source"],
    values="TWh",
    color="Source",
    color_discrete_map={
        "Coal": "#E69F00",
        "Oil": "#56B4E9",
        "Natural Gas": "#009E73"
    },
    title="China shows the highest fossil fuel dependence in TWh"
)

fig.update_traces(root_color="lightgrey")

fig.show()

## Task 2 — Treemap vs bar: low-carbon energy by country

**What to build:** Build **both** a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `'Low-carbon'`
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
- Both charts show TWh values, not percentages
- Insight title on the bar chart naming the leading country

In [6]:
# Task 2 — charts
# YOUR CODE HERE
df_low = (
    df[df["Source_Type"] == "Low-carbon"]
    .groupby("Country", as_index=False)["TWh"]
    .sum()
)

df_low["Root"] = "Low-carbon"

fig1 = px.treemap(
    df_low,
    path=["Root", "Country"],
    values="TWh",
    title="Low-carbon energy by country"
)

fig1.show()

fig2 = px.bar(
    df_low.sort_values("TWh"),
    x="TWh",
    y="Country",
    orientation="h",
    color="TWh",
    title=f"{df_low.loc[df_low['TWh'].idxmax(),'Country']} leads low-carbon energy production"
)

fig2.show()

### Task 2 — reflection

**Which chart tells the ranking story more clearly, and why?**

The bar chart shows country rankings more clearly because countries can be compared along a common axis. The treemap is better for showing part-to-whole relationships.

## Task 3 — 100% stacked bar: which region relies most on coal?

**What to build:** A horizontal 100% stacked bar chart showing, for each **Region**, how its fossil energy splits between Coal, Oil and Natural Gas. This is the **same data as your Task 1 treemap**, visualised a second way.

**Requirements:**
- Reuse `df_fossil` from Task 1; aggregate TWh by `Region` × `Source` and convert to **shares of each region's fossil total**
- Horizontal bars, regions **sorted by coal share** (highest at the top)
- Coal is the segment on the baseline (first in `category_orders`) and the **only coloured segment** — keep it the same colour as in Task 1; Oil and Natural Gas in two greys
- Percentage labels on the coal segments only
- Insight title naming the most coal-dependent region and its coal share

> 💡 `df.groupby('Region')['TWh'].transform('sum')` gives each row its region total

In [7]:
# Task 3
# YOUR CODE HERE
regional = (
    df_fossil.groupby(["Region", "Source"], as_index=False)["TWh"]
    .sum()
)

regional["Region_Total"] = regional.groupby("Region")["TWh"].transform("sum")
regional["Share"] = regional["TWh"] / regional["Region_Total"] * 100

coal_order = (
    regional[regional["Source"] == "Coal"]
    .sort_values("Share", ascending=False)["Region"]
)

fig = px.bar(
    regional,
    x="Share",
    y="Region",
    color="Source",
    orientation="h",
    category_orders={"Region": coal_order.tolist()},
    title="Asia has the highest coal dependence among regions"
)

fig.show()

### Task 3 — reflection

**Your Task 1 treemap and this stacked bar show the same fossil data. Name one question each chart answers that the other cannot.**

The treemap answers which countries contribute the most fossil energy overall. The stacked bar answers how each region splits its fossil energy between coal, oil and natural gas.